In [35]:
import h5py


H5_PATHs = ["../outputs/results/simulation_results_INSIDE_fgp_tabu_global_benchmark_[8]pzs_[3]perpz_WITHDAG.h5", "../outputs/results/simulation_results_fgp_tabu_global_benchmark_2pzs_6perpz_WITHDAG.h5", "../outputs/results/simulation_results_fgp_tabu_global_benchmark_4pzs_3perpz_NODAG.h5"]

for H5_PATH in H5_PATHs:
    with h5py.File(H5_PATH, "r") as f:
        results = f.get("results")
        if results is None:
            raise ValueError("No 'results' group found in H5 file")

        run_names = list(results.keys())
        print(f"Runs: {len(run_names)}")
        for run_name in run_names[:5]:
            attrs = dict(results[run_name].attrs)
            print(run_name, attrs)


Runs: 24
run_0000 {'algorithm_name': 'qft_nativegates_quantinuum_qiskit_opt2', 'cost_after': np.float64(nan), 'cost_before': np.float64(nan), 'cpu_time_seconds': np.float64(105.101692), 'enable_memory_zone_manager': np.False_, 'enforce_slice_plan': np.False_, 'final_timesteps': np.int64(2317), 'gate_count_1q': np.int64(902), 'gate_count_2q': np.int64(410), 'grid_size': np.int64(4), 'ions_per_pz': np.int64(3), 'move_distance_total': np.float64(nan), 'mz_trap_size': np.int64(1), 'num_ions': np.int64(20), 'num_pzs': np.int64(8), 'optimize_params': np.False_, 'partitioning_algorithm': 'none', 'plot': np.False_, 'save': np.False_, 'success': np.True_, 'timesteps_lower_bound': np.int64(266), 'use_dag': np.False_}
run_0001 {'algo_balance_penalty': np.int64(5), 'algo_candidate_list_length': 'None', 'algo_distance_weight_factor': np.int64(1), 'algo_max_iterations_factor': np.int64(100), 'algo_seed': np.int64(0), 'algorithm_name': 'qft_nativegates_quantinuum_qiskit_opt2', 'cost_after': np.float6

In [36]:
import csv
import h5py



CSV_PATH = "../outputs/results/simulation_results_fgp_tabu_global_benchmark_combined.csv"

columns = [
    "algorithm_name",
    "partitioning_algorithm",
    "gate_count_1q",
    "gate_count_2q",
    "num_ions",
    "gate_count",
    "success",
    "final_timesteps",
    "cpu_time_seconds",
    "idle_count",
    "gate_time_one_qubit",
    "gate_time_two_qubit",
    "num_pzs",
    "max_ions_per_pz",
    "arch",
]

with open(CSV_PATH, "w", newline="", encoding="utf-8") as csvfile:
    
    writer = csv.DictWriter(csvfile, fieldnames=columns)
    writer.writeheader()

    for H5_PATH in H5_PATHs:
    

        with h5py.File(H5_PATH, "r") as f:
            results = f.get("results")
            if results is None:
                raise ValueError("No 'results' group found in H5 file")

        
            for run_name in results.keys():
                attrs = results[run_name].attrs
                row = {key: attrs.get(key) for key in columns}
                writer.writerow(row)

print(f"Wrote {CSV_PATH}")


Wrote ../outputs/results/simulation_results_fgp_tabu_global_benchmark_combined.csv


In [45]:
import csv
from collections import defaultdict


CSV_PATH = "../outputs/results/simulation_results_fgp_tabu_global_benchmark_combined.csv"
LATEX_TXT_PATH = "../outputs/results/fgp_latex_table.txt"

BENCHMARK_LABELS = {
    "qft": "QFT",
    "qpeexact": "QPEexact",
    "qaoa": "QAOA",
    "random": "Random",
}
BENCHMARK_ORDER = ["QFT", "QPEexact", "QAOA", "Random"]


def normalize_benchmark(name: str) -> str:
    base = name.split("nativegates")[0].rstrip("_")
    return BENCHMARK_LABELS.get(base, base)


def to_int(value):
    if value in (None, ""):
        return None
    return int(value)


def to_float(value):
    if value in (None, ""):
        return None
    return float(value)


# ----------------------------
# CSV aggregation
# ----------------------------

rows = {}

with open(CSV_PATH, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        architecture = (row.get("num_pzs") or "").strip()
        benchmark = normalize_benchmark(row.get("algorithm_name", ""))
        qubits = to_int(row.get("num_ions"))

        if architecture == "" or benchmark == "" or qubits is None:
            continue

        key = (architecture, benchmark, qubits)

        entry = rows.setdefault(
            key,
            {
                "architecture": architecture,
                "benchmark": benchmark,
                "qubits": qubits,
                "g1": None,
                "g2": None,
                "t_static": None,
                "t_fgp": None,
                "dt": None,
                "cpu_static": None,
                "cpu_fgp": None,
            },
        )

        g1 = to_int(row.get("gate_count_1q"))
        g2 = to_int(row.get("gate_count_2q"))
        if entry["g1"] is None and g1 is not None:
            entry["g1"] = g1
        if entry["g2"] is None and g2 is not None:
            entry["g2"] = g2

        timesteps = to_int(row.get("final_timesteps"))
        cpu = to_float(row.get("cpu_time_seconds"))

        part = (row.get("partitioning_algorithm") or "").strip().lower()
        if part == "none":
            entry["t_static"] = timesteps
            entry["cpu_static"] = cpu
        elif part == "fgp_tabu_global":
            entry["t_fgp"] = timesteps
            entry["cpu_fgp"] = cpu


# ----------------------------
# Derived quantities
# ----------------------------

for entry in rows.values():
    if entry["cpu_fgp"] is not None and entry["cpu_static"] is not None:
        entry["dt"] = entry["cpu_fgp"] - entry["cpu_static"]
    else:
        entry["dt"] = None

    
for entry in rows.values():
    if entry["t_fgp"] is not None and entry["t_static"] not in (None, 0):
        entry["pct_timesteps"] = (
            100.0 * (entry["t_fgp"] - entry["t_static"]) / entry["t_static"]
        )
    else:
        entry["pct_timesteps"] = None

    if entry["cpu_fgp"] is not None and entry["cpu_static"] not in (None, 0):
        entry["pct_cpu"] = (
            100.0 * (entry["cpu_fgp"] - entry["cpu_static"]) / entry["cpu_static"]
        )
    else:
        entry["pct_cpu"] = None



# ----------------------------
# Formatting helpers
# ----------------------------

def fmt_int(value):
    return "-" if value is None else f"{int(value)}"


def fmt_float(value):
    return "-" if value is None else f"{value:.2f}"


def fmt_fgp_with_delta(t_fgp, t_static):
    if t_fgp is None:
        return "-"
    if t_static is None or t_static == 0:
        return f"{int(t_fgp)}"

    delta = 100.0 * (t_fgp - t_static) / t_static
    sign = "+" if delta > 0 else ""
    perc = f"{sign}{delta:.1f}\\%"

    if delta < 0:
        perc = f"\\textcolor{{Green}}{{\\textbf{{{perc}}}}}"
    elif delta > 0:
        perc = f"\\textcolor{{Red}}{{{perc}}}"

    return f"{int(t_fgp)} ({perc})"



def fmt_dt_with_delta(dt, cpu_static):
    if dt is None:
        return "-"
    if cpu_static is None or cpu_static == 0:
        return f"{dt:.2f}"

    delta_pct = 100.0 * dt / cpu_static
    sign = "+" if delta_pct > 0 else ""
    perc = f"{sign}{delta_pct:.1f}\\%"

    if delta_pct < 0:
        perc = f"\\textbf{{{perc}}}"

    return f"{dt:.2f} ({perc})"

def fmt_avg_pct(pct):
    if pct is None:
        return "-"
    sign = "+" if pct > 0 else ""
    txt = f"{sign}{pct:.1f}\\%"

    if pct < 0:
        txt = f"\\textcolor{{Green}}{{\\textbf{{{txt}}}}}"
    elif pct > 0:
        txt = f"\\textcolor{{Red}}{{{txt}}}"

    return txt


# ----------------------------
# Sorting
# ----------------------------

def sort_key(item):
    architecture, benchmark, qubits = item[0]
    bench_order = (
        BENCHMARK_ORDER.index(benchmark)
        if benchmark in BENCHMARK_ORDER
        else len(BENCHMARK_ORDER)
    )
    return (architecture, bench_order, benchmark, qubits)


sorted_items = sorted(rows.items(), key=sort_key)



arch_stats = defaultdict(lambda: {
    "pct_timesteps": [],
    "pct_cpu": [],
})

for (_, _, _), entry in rows.items():
    arch = entry["architecture"]
    if entry["pct_timesteps"] is not None:
        arch_stats[arch]["pct_timesteps"].append(entry["pct_timesteps"])
    if entry["pct_cpu"] is not None:
        arch_stats[arch]["pct_cpu"].append(entry["pct_cpu"])

arch_avg = {}
for arch, vals in arch_stats.items():
    arch_avg[arch] = {
        "pct_timesteps": (
            sum(vals["pct_timesteps"]) / len(vals["pct_timesteps"])
            if vals["pct_timesteps"] else None
        ),
        "pct_cpu": (
            sum(vals["pct_cpu"]) / len(vals["pct_cpu"])
            if vals["pct_cpu"] else None
        ),
    }


# ----------------------------
# LaTeX generation
# ----------------------------

lines = []
lines.append("\\begin{table*}[t]")
lines.append("\\centering")
lines.append("\\caption{Comparison of static and fine-grained partitioning across benchmarks.}")
lines.append("\\label{tab:benchmark_results}")
lines.append("\\begin{tabular}{lllrr|r|r|r}")
lines.append("\\hline")
lines.append(
    "Architecture & Benchmark & Qubits & "
    "\\#1Q Gates & \\#2Q Gates & "
    "Timesteps Static & Timesteps FGP & $\\Delta t$ [$s$] \\\\"
)
lines.append("\\hline")

last_arch = None
last_benchmark = None

for _, entry in sorted_items:
    if last_arch is not None and entry["architecture"] != last_arch:
        avg = arch_avg[last_arch]
        lines.append(
            "\\hline\n"
            f"\\textbf{{$\\varnothing$}} & "
            f"\\multicolumn{{4}}{{r}}{{}} & | &"
            f"{fmt_avg_pct(avg['pct_timesteps'])} & "
            f"{fmt_avg_pct(avg['pct_cpu'])} \\\\"
        )
        lines.append("\\hline\\hline")

    elif last_benchmark is not None and entry["benchmark"] != last_benchmark:
        lines.append("\\hline")

    line = (
        "{arch} & {benchmark} & {qubits} & {g1} & {g2} & "
        "{t_static} & {t_fgp} & {dt} \\\\"
    ).format(
        arch=entry["architecture"],
        benchmark=entry["benchmark"],
        qubits=fmt_int(entry["qubits"]),
        g1=fmt_int(entry["g1"]),
        g2=fmt_int(entry["g2"]),
        t_static=fmt_int(entry["t_static"]),
        t_fgp=fmt_fgp_with_delta(entry["t_fgp"], entry["t_static"]),
        dt=fmt_dt_with_delta(entry["dt"], entry["cpu_static"]),
    )

    lines.append(line)
    last_arch = entry["architecture"]
    last_benchmark = entry["benchmark"]

if last_arch is not None:
    avg = arch_avg[last_arch]
    lines.append(
        "\\hline\n"
        f"\\textbf{{$\\varnothing$}} & "
        f"\\multicolumn{{4}}{{r}}{{}} & | &"
        f"{fmt_avg_pct(avg['pct_timesteps'])} & "
        f"{fmt_avg_pct(avg['pct_cpu'])} \\\\"
    )
    lines.append("\\hline")


lines.append("\\hline")

lines.append("\\end{tabular}")
lines.append("\\end{table*}")

latex = "\n".join(lines)

print(latex)

with open(LATEX_TXT_PATH, "w", encoding="utf-8") as f:
    f.write(latex)


\begin{table*}[t]
\centering
\caption{Comparison of static and fine-grained partitioning across benchmarks.}
\label{tab:benchmark_results}
\begin{tabular}{lllrr|r|r|r}
\hline
Architecture & Benchmark & Qubits & \#1Q Gates & \#2Q Gates & Timesteps Static & Timesteps FGP & $\Delta t$ [$s$] \\
\hline
2 & QFT & 20 & - & - & 3080 & 2125 (\textcolor{Green}{\textbf{-31.0\%}}) & 10.09 (+72.3\%) \\
2 & QFT & 30 & - & - & 6946 & 5558 (\textcolor{Green}{\textbf{-20.0\%}}) & 39.93 (+103.5\%) \\
2 & QFT & 40 & - & - & 11376 & 9009 (\textcolor{Green}{\textbf{-20.8\%}}) & 111.42 (+102.3\%) \\
2 & QFT & 50 & - & - & 16732 & 13312 (\textcolor{Green}{\textbf{-20.4\%}}) & 98.08 (+39.6\%) \\
\hline
2 & QPEexact & 20 & - & - & 2897 & 2285 (\textcolor{Green}{\textbf{-21.1\%}}) & 23.96 (+206.9\%) \\
2 & QPEexact & 30 & - & - & 5864 & 5075 (\textcolor{Green}{\textbf{-13.5\%}}) & 56.91 (+158.5\%) \\
2 & QPEexact & 40 & - & - & 11335 & 8231 (\textcolor{Green}{\textbf{-27.4\%}}) & 92.87 (+98.6\%) \\
2 & QPEexact